In [7]:
import math
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer
from collections import defaultdict


nltk.download('punkt')
nltk.download('stopwords')
nltk.download('punkt_tab')

# Initialize NLP tools
stemmer = PorterStemmer()
stop_words = set(stopwords.words('english'))

def preprocess(text):
    """Tokenizes, removes stop words, and applies Porter Stemming."""
    # Tokenize and convert to lowercase
    tokens = word_tokenize(text.lower())

    processed_tokens = [
        stemmer.stem(w) for w in tokens
        if w.isalnum() and w not in stop_words
    ]
    return processed_tokens


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [8]:
# Corpus 1
corpus_1 = {
    "D1": "Cats are domestic animals.",
    "D2": "Dogs are known for loyalty.",
    "D3": "Cats and dogs are both popular pets.",
    "D4": "Birds are free animals.",
    "D5": "The loyalty of cats is underrated.",
    "D6": "Pets like cats and dogs require care."
}

query_1 = "loyals pets"
processed_query_1 = preprocess(query_1)
print(f"Processed Query 1: {processed_query_1}\n")

# Preprocess Corpus 1
processed_corpus_1 = {doc_id: preprocess(text) for doc_id, text in corpus_1.items()}

# Determine Relevance (R vs NR)
relevant_docs_1 = set()
for doc_id, tokens in processed_corpus_1.items():
    if any(term in tokens for term in processed_query_1):
        relevant_docs_1.add(doc_id)

N = len(corpus_1)
R = len(relevant_docs_1)

# Calculate RSJ Weights for each query term
rsj_weights = {}
for term in processed_query_1:
    # n = total docs with term
    docs_with_term = [doc_id for doc_id, tokens in processed_corpus_1.items() if term in tokens]
    n = len(docs_with_term)

    # r = relevant docs with term
    r = len([doc_id for doc_id in docs_with_term if doc_id in relevant_docs_1])

    # RSJ Formula
    numerator = (r + 0.5) / (R - r + 0.5)
    denominator = (n - r + 0.5) / (N - n - R + r + 0.5)
    weight = math.log10(numerator / denominator)
    rsj_weights[term] = weight

# Score and Rank Documents
scores_1 = defaultdict(float)
for doc_id, tokens in processed_corpus_1.items():
    for term in processed_query_1:
        if term in tokens:
            scores_1[doc_id] += rsj_weights[term]

print(" RSJ Document Ranking ")
for doc_id, score in sorted(scores_1.items(), key=lambda x: x[1], reverse=True):
    print(f"{doc_id}: Score = {score:.3f}")

Processed Query 1: ['loyal', 'pet']

 RSJ Document Ranking 
D3: Score = 1.653
D6: Score = 1.653


In [9]:
# Corpus 2
corpus_2 = {
    "D1": "Dogs are known to be loyal pets.",
    "D2": "Cats are often kept as cute furry companions.",
    "D3": "The tech industry has seen a rise in AI innovations.",
    "D4": "Dogs and cats can both be trained.",
    "D5": "The smartphone market is dominated by tech giants.",
    "D6": "Goldfish are popular pets in many households.",
    "D7": "Innovations in robotics have changed industries.",
    "D8": "Pets offer comfort and emotional support to their owners.",
    "D9": "Owning a pet requires responsibility.",
    "D10": "AI-driven robots are the future of tech."
}

query_2 = "Pets and techs and innovations"
processed_query_2 = preprocess(query_2)
print(f"Processed Query 2: {processed_query_2}\n")

# Preprocess Corpus 2
processed_corpus_2 = {doc_id: preprocess(text) for doc_id, text in corpus_2.items()}

# Determine Relevance (R vs NR)
relevant_docs_2 = set()
for doc_id, tokens in processed_corpus_2.items():
    if any(term in tokens for term in processed_query_2):
        relevant_docs_2.add(doc_id)

N2 = len(corpus_2)
R2 = len(relevant_docs_2)
NR2 = N2 - R2

# Calculate Probabilities and Weights
prob_weights = {}
for term in processed_query_2:
    docs_with_term = [doc_id for doc_id, tokens in processed_corpus_2.items() if term in tokens]

    # Count occurrences in R and NR sets
    r_count = sum(1 for doc_id in docs_with_term if doc_id in relevant_docs_2)
    nr_count = sum(1 for doc_id in docs_with_term if doc_id not in relevant_docs_2)

    # Calculate Probabilities
    p_t_R = r_count / R2 if R2 > 0 else 0
    p_t_NR = nr_count / NR2 if NR2 > 0 else 0

    # Apply 0.01 Smoothing if necessary
    p_t_R = 0.01 if p_t_R == 0 else p_t_R
    p_t_NR = 0.01 if p_t_NR == 0 else p_t_NR

    # Calculate Term Weight
    prob_weights[term] = math.log10(p_t_R / p_t_NR)

# Score and Rank Documents
scores_2 = defaultdict(float)
for doc_id, tokens in processed_corpus_2.items():
    for term in processed_query_2:
        if term in tokens:
            scores_2[doc_id] += prob_weights[term]

print(" Probabilistic Document Ranking (Smoothed) ")
for doc_id, score in sorted(scores_2.items(), key=lambda x: x[1], reverse=True):
    print(f"{doc_id}: Score = {score:.3f}")

Processed Query 2: ['pet', 'tech', 'innov']

 Probabilistic Document Ranking (Smoothed) 
D3: Score = 2.972
D1: Score = 1.699
D6: Score = 1.699
D8: Score = 1.699
D9: Score = 1.699
D5: Score = 1.574
D10: Score = 1.574
D7: Score = 1.398
